# Content-Style Cluster Prediction with Random Forest

Train a Random Forest classifier to predict content-style cluster labels (from DBSCAN)
using content-style features from `reports/all_posts_clustered.csv`.

In [86]:
from pathlib import Path

import joblib
import numpy as np
import pandas as pd
from sklearn.linear_model import LogisticRegression
from sklearn.metrics import (
    accuracy_score,
    classification_report,
    confusion_matrix,
)
from sklearn.model_selection import GroupKFold, GroupShuffleSplit, cross_val_score
from sklearn.pipeline import Pipeline
from sklearn.compose import ColumnTransformer
from sklearn.preprocessing import OneHotEncoder, StandardScaler

print("Imports ready.")


Imports ready.


In [87]:
# Load clustered posts CSV
data_path_candidates = [
    Path("notebooks/clustering/reports/all_posts_clustered.csv"),
    Path("reports/all_posts_clustered.csv"),
]
DATA_PATH = next((pp for pp in data_path_candidates if pp.exists()), data_path_candidates[0])
df = pd.read_csv(DATA_PATH)
print(f"Loaded from: {DATA_PATH}")
print(f"Total posts: {len(df)}")
print(f"Columns: {df.columns.tolist()}")

LABEL_COL = "cluster_id"
GROUP_COL = "business_name"

NUMERIC_COLS = [
    "likes_count",
    "comments_count",
    "views_count",
    "engagement_rate",
    "view_rate",
]

BINARY_COLS = [
    "CTA_present",
    "promo_post",
    "mentions_location",
    "religious_theme",
    "patriotic_theme",
    "arabic_dialect_style",
]

CATEGORICAL_COLS = [
    "sector",
    "post_type",
]

FEATURE_COLS = NUMERIC_COLS + BINARY_COLS + CATEGORICAL_COLS

print(f"
Label column: {LABEL_COL}")
print(f"Group column: {GROUP_COL}")
print(f"Numeric cols (scaled): {NUMERIC_COLS}")
print(f"Binary cols: {BINARY_COLS}")
print(f"Categorical cols (one-hot): {CATEGORICAL_COLS}")
print(f"
Label distribution:
{df[LABEL_COL].value_counts().sort_index()}")


Total posts: 775
Columns: ['cluster_id', 'business_name', 'sector', 'post_type', 'caption_text', 'likes_count', 'comments_count', 'views_count', 'engagement_rate', 'view_rate', 'CTA_present', 'promo_post', 'mentions_location', 'religious_theme', 'patriotic_theme', 'arabic_dialect_style']

Feature columns: ['CTA_present', 'promo_post', 'mentions_location', 'religious_theme', 'patriotic_theme', 'arabic_dialect_style']
Label column: cluster_id

Label distribution:
cluster_id
-1    317
 0     57
 1     38
 2    211
 3     66
 4     86
Name: count, dtype: int64


In [88]:
# Prepare X (structured features) and y (labels)
X = df[FEATURE_COLS].copy()
y = df[LABEL_COL].copy()

# Filter out noise (label == -1)
mask = y != -1
X_clean = X.loc[mask].copy()
y_clean = y.loc[mask].copy()
groups_clean = df.loc[mask, GROUP_COL].astype(str).fillna("unknown")

# Clean numeric/binary
for c in NUMERIC_COLS:
    X_clean[c] = pd.to_numeric(X_clean[c], errors="coerce").fillna(0.0)
for c in BINARY_COLS:
    X_clean[c] = pd.to_numeric(X_clean[c], errors="coerce").fillna(0).astype(int)

# Clean categorical
for c in CATEGORICAL_COLS:
    X_clean[c] = X_clean[c].astype(str).fillna("unknown")

print(f"Before filtering noise: {len(X)} posts")
print(f"After  filtering noise: {len(X_clean)} posts ({len(X_clean) / len(X) * 100:.1f}% retained)")
print(f"
Clean label distribution:
{y_clean.value_counts().sort_index()}")


Before filtering noise: 775 posts
After  filtering noise: 458 posts (59.1% retained)

Clean label distribution:
cluster_id
0     57
1     38
2    211
3     66
4     86
Name: count, dtype: int64


In [89]:
# Diagnostic: duplicate structured rows and their label ambiguity
diag = X_clean.copy()
diag[LABEL_COL] = y_clean.values

dup_map = diag.groupby(FEATURE_COLS)[LABEL_COL].nunique().reset_index(name="n_labels")
ambiguous = (dup_map["n_labels"] > 1).sum()
deterministic = (dup_map["n_labels"] == 1).all()

print(f"Unique structured feature rows: {len(dup_map)}")
print(f"Rows mapping to >1 label: {ambiguous}")
if deterministic:
    print("WARNING: Deterministic structured-feature -> label mapping detected.")
else:
    print("No deterministic mapping across full structured features.")


Unique feature combos: 5
Ambiguous combos (>1 label): 0


In [90]:
# Grouped train / test split by business_name (prevents same business in train and test)
gss = GroupShuffleSplit(n_splits=1, test_size=0.2, random_state=42)
train_idx, test_idx = next(gss.split(X_clean, y_clean, groups=groups_clean))

X_train = X_clean.iloc[train_idx].copy()
X_test = X_clean.iloc[test_idx].copy()
y_train = y_clean.iloc[train_idx].copy()
y_test = y_clean.iloc[test_idx].copy()
groups_train = groups_clean.iloc[train_idx].copy()
groups_test = groups_clean.iloc[test_idx].copy()

overlap = set(groups_train.unique()) & set(groups_test.unique())
print(f"Train set: {len(X_train)} posts")
print(f"Test set:  {len(X_test)} posts")
print(f"Train businesses: {groups_train.nunique()}")
print(f"Test businesses:  {groups_test.nunique()}")
print(f"Business overlap train/test: {len(overlap)}")


Train set: 347 posts
Test set:  111 posts
Train businesses: 40
Test businesses:  10
Business overlap train/test: 0


In [92]:
# Train structured-feature pipeline (scale numeric, one-hot categorical)
preprocess = ColumnTransformer([
    ("num", StandardScaler(), NUMERIC_COLS),
    ("bin", "passthrough", BINARY_COLS),
    ("cat", OneHotEncoder(handle_unknown="ignore"), CATEGORICAL_COLS),
], remainder="drop")

pipe = Pipeline([
    ("prep", preprocess),
    ("clf", LogisticRegression(max_iter=4000, random_state=42, class_weight="balanced")),
])

pipe.fit(X_train, y_train)
print("Structured-feature Logistic Regression pipeline trained.")


Logistic Regression trained.


In [102]:
# Grouped cross-validation by business_name
cv = GroupKFold(n_splits=3)
cv_scores = cross_val_score(pipe, X_clean, y_clean, cv=cv, groups=groups_clean, scoring="accuracy")
print(f"Grouped CV accuracy scores: {cv_scores}")
print(f"Mean CV accuracy:  {cv_scores.mean():.4f} (+/- {cv_scores.std() * 2:.4f})")


Best params: {'C': 10, 'class_weight': None, 'max_iter': 2000, 'penalty': 'l2', 'solver': 'lbfgs'}
Best grouped CV accuracy: 1.0


c:\Users\hanib\AppData\Local\Programs\Python\Python313\Lib\site-packages\sklearn\linear_model\_logistic.py:1135: FutureWarning: 'penalty' was deprecated in version 1.8 and will be removed in 1.10. To avoid this warning, leave 'penalty' set to its default value and use 'l1_ratio' or 'C' instead. Use l1_ratio=0 instead of penalty='l2', l1_ratio=1 instead of penalty='l1', and C=np.inf instead of penalty=None.
  warnings.warn(


In [99]:
# Evaluate on test set
y_pred = pipe.predict(X_test)

acc = accuracy_score(y_test, y_pred)
print(f"Test accuracy: {acc:.4f}")
print(f"\nClassification Report:")
print(classification_report(y_test, y_pred, zero_division=0))
print(f"Confusion Matrix:")
print(confusion_matrix(y_test, y_pred))


Test accuracy: 1.0000

Classification Report:
              precision    recall  f1-score   support

           0       1.00      1.00      1.00        15
           1       1.00      1.00      1.00        14
           2       1.00      1.00      1.00        34
           3       1.00      1.00      1.00        28
           4       1.00      1.00      1.00        20

    accuracy                           1.00       111
   macro avg       1.00      1.00      1.00       111
weighted avg       1.00      1.00      1.00       111

Confusion Matrix:
[[15  0  0  0  0]
 [ 0 14  0  0  0]
 [ 0  0 34  0  0]
 [ 0  0  0 28  0]
 [ 0  0  0  0 20]]


In [95]:
# Top weighted transformed features per class
prep = pipe.named_steps["prep"]
clf = pipe.named_steps["clf"]
feat_names = prep.get_feature_names_out()

for i, cls in enumerate(clf.classes_):
    coefs = clf.coef_[i]
    top_idx = np.argsort(coefs)[-10:][::-1]
    print(f"
Class {cls} top positive features:")
    for j in top_idx:
        print(f"  {feat_names[j]}: {coefs[j]:.4f}")


             feature  coef_abs_mean
         CTA_present       1.982675
          promo_post       1.607421
arabic_dialect_style       1.607421
   mentions_location       1.542905
     religious_theme       0.000000
     patriotic_theme       0.000000


In [96]:
# Save model pipeline
MODEL_PATH = Path("artifacts/content_style_clustering/models/content_style_structured_logreg_pipeline.joblib")
MODEL_PATH.parent.mkdir(parents=True, exist_ok=True)
joblib.dump(pipe, MODEL_PATH)
print(f"Model saved to: {MODEL_PATH}")


Model saved to: artifacts\content_style_clustering\models\content_style_logreg_model.joblib
Feature columns saved to: artifacts\content_style_clustering\models\feature_cols.joblib


In [ ]:
# Prediction function
def predict_content_style_cluster(posts_df: pd.DataFrame, model_path: str = None) -> pd.DataFrame:
    """
    Predict content-style cluster label for each post using structured-feature Logistic Regression pipeline.

    Args:
        posts_df: DataFrame with required structured feature columns.
        model_path: Path to saved pipeline joblib. Uses default if None.

    Returns:
        DataFrame with predicted cluster labels appended as predicted_cluster column.
    """
    model_path = Path(model_path or "artifacts/content_style_clustering/models/content_style_structured_logreg_pipeline.joblib")
    model = joblib.load(model_path)

    X_in = posts_df.copy()
    for c in FEATURE_COLS:
        if c not in X_in.columns:
            X_in[c] = 0 if c in (NUMERIC_COLS + BINARY_COLS) else "unknown"

    for c in NUMERIC_COLS:
        X_in[c] = pd.to_numeric(X_in[c], errors="coerce").fillna(0.0)
    for c in BINARY_COLS:
        X_in[c] = pd.to_numeric(X_in[c], errors="coerce").fillna(0).astype(int)
    for c in CATEGORICAL_COLS:
        X_in[c] = X_in[c].astype(str).fillna("unknown")

    out = posts_df.copy()
    out["predicted_cluster"] = model.predict(X_in[FEATURE_COLS])
    return out

print("predict_content_style_cluster function defined.")


predict_content_style_cluster function defined.


In [ ]:
# Quick test: predict on a sample
sample_df = X_test.head(5).copy()
pred = predict_content_style_cluster(sample_df)
actual = y_test.head(5).values
print("Actual vs Predicted:")
for i in range(len(sample_df)):
    print(f"  Actual: {actual[i]}, Predicted: {pred['predicted_cluster'].iloc[i]}, Match: {actual[i] == pred['predicted_cluster'].iloc[i]}")

Actual vs Predicted:
  Actual: 0, Predicted: 0, Match: True
  Actual: 0, Predicted: 0, Match: True
  Actual: 0, Predicted: 0, Match: True
  Actual: 0, Predicted: 0, Match: True
  Actual: 0, Predicted: 0, Match: True
